# ĐỒ ÁN 1: MA TRẬN VÀ CỞ SỞ CỦA TÍNH TOÁN KHOA HỌC
## PHẦN 1: PHÉP KHỬ GAUSS VÀ CÁC ỨNG DỤNG

**Trường:** Đại học Khoa học Tự nhiên - ĐHQG HCM
**Môn học:** Toán Ứng Dụng và Thống Kê  

---
**Tóm tắt:** Notebook này trình bày các kết quả thử nghiệm của việc cài đặt thuật toán khử Gauss có chọn phần tử chốt (Partial Pivoting) từ đầu (from scratch) bằng Python. Các module được tách thành 4 file `.py` riêng biệt và được gọi vào đây để đối chiếu kết quả với thư viện `NumPy`.

In [1]:
import numpy as np

# Import các module của nhóm
from gaussian import gaussian_eliminate, general_solution, verify_solution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("✅ Đã load thành công thư viện và các module của Nhóm 10!")

✅ Đã load thành công thư viện và các module của Nhóm 10!


## 1. Giải hệ phương trình tuyến tính (Gauss Elimination)
Kiểm tra khả năng giải hệ phương trình $Ax = b$ trong các trường hợp: nghiệm duy nhất, vô số nghiệm và kết hợp đối chiếu sai số với NumPy.

## 2. Tính định thức (Determinant)
Kiểm tra thuật toán tính định thức dựa trên số lần hoán đổi dòng ($s$) trong phép khử Gauss.

## 3. Tìm ma trận nghịch đảo (Inverse Matrix)
Sử dụng thuật toán Gauss-Jordan trên ma trận mở rộng $[A | I]$. Để kiểm chứng, ta sẽ nhân ma trận gốc $A$ với ma trận nghịch đảo $A^{-1}$ tìm được để xem có ra ma trận đơn vị $I$ hay không.

In [4]:
np.random.seed(42)

cases_inv = [
    ("Ma trận 3x3 khả nghịch", [[2, 1, 1], [1, 3, 2], [1, 0, 0]]),
    ("Ma trận có phần tử A[0][0] = 0", [[0, 2, 3], [1, 1, -1], [2, -1, 1]]),
    ("Ma trận suy biến (det = 0)", [[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
    ("Ma trận chứa số cực nhỏ", [[1e-10, 1], [1, 1]]),
    ("Ma trận không vuông (Bắt lỗi)", [[1, 2, 3], [4, 5, 6]]),
    ("Ma trận ngẫu nhiên (50x50)", np.random.rand(50, 50).tolist())
]

for i, (desc, A) in enumerate(cases_inv, 1):
    print(f"\n--- Test {i}: {desc} ---")
    invA = inverse(A)
    
    if invA is not None:
        A_np, invA_np = np.array(A), np.array(invA)
        
        err_identity = float(np.linalg.norm((A_np @ invA_np) - np.eye(len(A)), ord=np.inf))
        err_numpy = float(np.linalg.norm(invA_np - np.linalg.inv(A_np), ord=np.inf))
        
        print(f"Sai số ||A * A^-1 - I||_inf : {err_identity:.2e}")
        print(f"Sai số ||A^-1 - NumPy||_inf : {err_numpy:.2e}")
    else:
        print("Kết quả: Không tồn tại ma trận nghịch đảo.")


--- Test 1: Ma trận 3x3 khả nghịch ---
Sai số ||A * A^-1 - I||_inf : 8.88e-16
Sai số ||A^-1 - NumPy||_inf : 2.00e-15

--- Test 2: Ma trận có phần tử A[0][0] = 0 ---
Sai số ||A * A^-1 - I||_inf : 1.11e-16
Sai số ||A^-1 - NumPy||_inf : 8.33e-17

--- Test 3: Ma trận suy biến (det = 0) ---
Lỗi: Ma trận suy biến (Singular Matrix), không thể tìm nghịch đảo.
Kết quả: Không tồn tại ma trận nghịch đảo.

--- Test 4: Ma trận chứa số cực nhỏ ---
Sai số ||A * A^-1 - I||_inf : 0.00e+00
Sai số ||A^-1 - NumPy||_inf : 0.00e+00

--- Test 5: Ma trận không vuông (Bắt lỗi) ---
Lỗi: Ma trận không vuông, không thể tìm nghịch đảo.
Kết quả: Không tồn tại ma trận nghịch đảo.

--- Test 6: Ma trận ngẫu nhiên (50x50) ---
Sai số ||A * A^-1 - I||_inf : 7.17e-14
Sai số ||A^-1 - NumPy||_inf : 2.38e-13


## 4. Hạng và cơ sở (Rank & Basis)
Từ ma trận bậc thang rút gọn (RREF), trích xuất hạng của ma trận và cơ sở của 3 không gian vector (Cột, Dòng, Nghiệm). So sánh hạng với hàm `matrix_rank` của NumPy.

In [5]:
np.random.seed(42)

cases_rank = [
    ("Ma trận 3x4 (Hạng 2)", [[1, 2, 0, 1], [2, 4, 1, 4], [3, 6, 1, 5]]),
    ("Ma trận vuông khả nghịch", [[2, 1, 1], [1, 3, 2], [1, 0, 0]]),
    ("Ma trận toàn số 0", [[0, 0, 0], [0, 0, 0]]),
    ("Ma trận có các dòng tỷ lệ (Hạng 1)", [[1, 2, 3], [2, 4, 6], [3, 6, 9]]),
    ("Ma trận đứng (4x2)", [[1, 2], [3, 4], [5, 6], [7, 8]]),
    ("Ma trận ngẫu nhiên chữ nhật (50x70)", np.random.rand(50, 70).tolist())
]

for i, (desc, A) in enumerate(cases_rank, 1):
    print(f"\n--- Test {i}: {desc} ---")
    res = rank_and_basis(A)
    
    A_np = np.array(A)
    rank_np = np.linalg.matrix_rank(A_np)
    
    n_cols = len(A[0]) if A else 0
    dim_col = len(res['column_space_basis'])
    dim_row = len(res['row_space_basis'])
    dim_null = len(res['null_space_basis'])
    
    print(f"Hạng (Tính toán / NumPy) : {res['rank']} / {rank_np}")
    print(f"Số chiều không gian      : Cột = {dim_col}, Dòng = {dim_row}, Nghiệm = {dim_null}")
    
    if dim_col + dim_null == n_cols:
        print(f"Định lý Rank-Nullity     : Hợp lệ ({dim_col} + {dim_null} = {n_cols})")
    else:
        print(f"Định lý Rank-Nullity     : Có sai sót!")


--- Test 1: Ma trận 3x4 (Hạng 2) ---
Hạng (Tính toán / NumPy) : 2 / 2
Số chiều không gian      : Cột = 2, Dòng = 2, Nghiệm = 2
Định lý Rank-Nullity     : Hợp lệ (2 + 2 = 4)

--- Test 2: Ma trận vuông khả nghịch ---
Hạng (Tính toán / NumPy) : 3 / 3
Số chiều không gian      : Cột = 3, Dòng = 3, Nghiệm = 0
Định lý Rank-Nullity     : Hợp lệ (3 + 0 = 3)

--- Test 3: Ma trận toàn số 0 ---
Hạng (Tính toán / NumPy) : 0 / 0
Số chiều không gian      : Cột = 0, Dòng = 0, Nghiệm = 3
Định lý Rank-Nullity     : Hợp lệ (0 + 3 = 3)

--- Test 4: Ma trận có các dòng tỷ lệ (Hạng 1) ---
Hạng (Tính toán / NumPy) : 1 / 1
Số chiều không gian      : Cột = 1, Dòng = 1, Nghiệm = 2
Định lý Rank-Nullity     : Hợp lệ (1 + 2 = 3)

--- Test 5: Ma trận đứng (4x2) ---
Hạng (Tính toán / NumPy) : 2 / 2
Số chiều không gian      : Cột = 2, Dòng = 2, Nghiệm = 0
Định lý Rank-Nullity     : Hợp lệ (2 + 0 = 2)

--- Test 6: Ma trận ngẫu nhiên chữ nhật (50x70) ---
Hạng (Tính toán / NumPy) : 50 / 50
Số chiều không gian      : Cộ